# Data Exploration & Cleaning - UCI Online Retail Dataset

**Portfolio Project:** Customer segmentation preparation - exploring and cleaning retail transaction data.

**Author:** Lee Christian Lesemann  
**Dataset:** UCI Machine Learning Repository - Online Retail Dataset  
**Objective:** Prepare high-quality data for RFM customer segmentation analysis

---

## Project Overview

This notebook performs comprehensive data exploration and cleaning on the UCI Online Retail dataset:
1. Initial data quality assessment
2. Missing value analysis
3. Outlier detection
4. Data cleaning pipeline
5. Export cleaned dataset for segmentation analysis

**Goal:** Transform 541k raw transactions into analysis-ready data while maintaining data integrity.

---

## 1. Environment Setup

Import required libraries and verify installation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Environment ready.")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version:  {np.__version__}")

## 2. Data Loading

Load the UCI Online Retail dataset from Excel format.

**Dataset Info:**
- Source: UCI Machine Learning Repository
- Format: Excel (.xlsx)
- Expected size: ~18 MB, 541k+ rows
- Columns: InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country

In [ ]:
# If the raw Excel file is missing, you can download it with:
#   from src.rfm_pipeline import download_dataset
#   download_dataset('../data')

file_path = '../data/online_retail.xlsx'

print("Loading data... (this may take ~30 seconds)")
df = pd.read_excel(file_path)

print(f"\nSuccessfully loaded.")
print(f"Rows:    {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"\nColumn names: {df.columns.tolist()}")

## 3. Initial Data Exploration

Examine dataset structure, data types, and basic statistics.

In [ ]:
# Display the first 10 rows
print("First 10 rows of the dataset:\n")
df.head(10)

In [ ]:
# Comprehensive data quality check
print("=" * 60)
print("DATA QUALITY CHECK")
print("=" * 60)

print(f"\nShape: {df.shape[0]:,} rows x {df.shape[1]} columns")

print(f"\nColumns and data types:")
print(df.dtypes)

print(f"\nMissing values:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing': missing,
    'Percent': missing_pct
})
print(missing_df[missing_df['Missing'] > 0])

print(f"\nDuplicates: {df.duplicated().sum():,}")

## 4. Data Quality Assessment

Identify data quality issues that need to be addressed:
- **Missing values:** 135k rows (25%) without CustomerID - cannot be segmented
- **Duplicates:** 5,268 duplicate transactions
- **Data types:** All columns correctly formatted

In [ ]:
# Descriptive statistics for numerical columns
print("DESCRIPTIVE STATISTICS")
print("=" * 60)
df.describe()

## 5. Statistical Overview

Analyze numerical distributions to identify potential issues:
- **Negative quantities:** Returns or data errors (min: -80,995)
- **Negative prices:** Data errors (min: -11,062)
- **Date range:** 12.4 months of data (Dec 2010 - Dec 2011)

In [ ]:
# Analyze the date range of the dataset
print("DATE RANGE ANALYSIS")
print("=" * 60)

# Ensure InvoiceDate is datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

date_span = df['InvoiceDate'].max() - df['InvoiceDate'].min()

print(f"First transaction:  {df['InvoiceDate'].min()}")
print(f"Last transaction:   {df['InvoiceDate'].max()}")
print(f"Time span:          {date_span.days} days ({date_span.days / 30:.1f} months)")

## 6. Data Cleaning Pipeline

Systematic cleaning process to ensure analysis-ready data.

**Cleaning Steps:**
1. Remove rows without CustomerID (cannot segment without customer identifier)
2. Remove cancelled orders (InvoiceNo starting with 'C')
3. Remove negative or zero quantities (returns/errors)
4. Remove negative or zero prices (data errors)
5. Create TotalAmount column (Quantity x UnitPrice)
6. Remove duplicates

**Note:** Outlier handling is deferred to the RFM analysis notebook (02), where it is applied
at the customer level after aggregation. Removing transaction-level outliers here would
prematurely discard valid high-value purchases before understanding per-customer distributions.

In [ ]:
# DATA CLEANING - STEP BY STEP

print("DATA CLEANING PIPELINE")
print("=" * 60)

# Original shape
original_rows = df.shape[0]
print(f"Original: {original_rows:,} rows")

# Step 1: Remove rows without CustomerID (cannot be segmented)
df_clean = df[df['CustomerID'].notna()].copy()
removed_no_customer = original_rows - df_clean.shape[0]
print(f"\nStep 1: Removed {removed_no_customer:,} rows without CustomerID")
print(f"   Remaining: {df_clean.shape[0]:,} rows")

# Step 2: Remove cancellations (InvoiceNo starts with 'C')
before = df_clean.shape[0]
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]
removed_cancellations = before - df_clean.shape[0]
print(f"\nStep 2: Removed {removed_cancellations:,} cancelled orders (InvoiceNo starting with 'C')")
print(f"   Remaining: {df_clean.shape[0]:,} rows")

# Step 3: Remove negative or zero quantities
before = df_clean.shape[0]
df_clean = df_clean[df_clean['Quantity'] > 0]
removed_negative_qty = before - df_clean.shape[0]
print(f"\nStep 3: Removed {removed_negative_qty:,} rows with Quantity <= 0")
print(f"   Remaining: {df_clean.shape[0]:,} rows")

# Step 4: Remove negative or zero prices
before = df_clean.shape[0]
df_clean = df_clean[df_clean['UnitPrice'] > 0]
removed_negative_price = before - df_clean.shape[0]
print(f"\nStep 4: Removed {removed_negative_price:,} rows with UnitPrice <= 0")
print(f"   Remaining: {df_clean.shape[0]:,} rows")

# Step 5: Create TotalAmount column (needed for Monetary value in RFM)
df_clean['TotalAmount'] = df_clean['Quantity'] * df_clean['UnitPrice']
print(f"\nStep 5: Added TotalAmount column (Quantity x UnitPrice)")

# Note: Outlier handling is done at the customer level in notebook 02
# after RFM aggregation. This preserves valid high-value transactions.

# Step 6: Remove duplicates
before = df_clean.shape[0]
df_clean = df_clean.drop_duplicates()
removed_duplicates = before - df_clean.shape[0]
print(f"\nStep 6: Removed {removed_duplicates:,} duplicate rows")
print(f"   Remaining: {df_clean.shape[0]:,} rows")

# Final summary
total_removed = original_rows - df_clean.shape[0]
print("\n" + "=" * 60)
print("CLEANING COMPLETE")
print("=" * 60)
print(f"Original:  {original_rows:,} rows")
print(f"Cleaned:   {df_clean.shape[0]:,} rows")
print(f"Removed:   {total_removed:,} rows ({total_removed / original_rows * 100:.1f}%)")
print(f"Retained:  {df_clean.shape[0] / original_rows * 100:.1f}%")
print(f"\nUnique customers: {df_clean['CustomerID'].nunique():,}")
print(f"Unique products:  {df_clean['StockCode'].nunique():,}")
print(f"Unique invoices:  {df_clean['InvoiceNo'].nunique():,}")

## 7. Export Cleaned Dataset

Save the cleaned data for use in the RFM analysis notebook.

In [ ]:
import os
import zipfile

# Save cleaned data as CSV
csv_path = '../data/online_retail_clean.csv'
df_clean.to_csv(csv_path, index=False)

csv_size_mb = os.path.getsize(csv_path) / 1024 / 1024
print(f"Saved cleaned dataset to: {csv_path}")
print(f"CSV file size: {csv_size_mb:.2f} MB")

# Compress to zip for storage efficiency
zip_path = '../data/online_retail_clean.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(csv_path, arcname='online_retail_clean.csv')

zip_size_mb = os.path.getsize(zip_path) / 1024 / 1024
print(f"Compressed to: {zip_path} ({zip_size_mb:.2f} MB)")

In [ ]:
# Customer-level insights from the cleaned dataset
print("CUSTOMER INSIGHTS")
print("=" * 60)

# Transactions per customer
transactions_per_customer = df_clean.groupby('CustomerID')['InvoiceNo'].nunique()
print(f"\nTransactions per customer:")
print(f"   Mean:   {transactions_per_customer.mean():.1f}")
print(f"   Median: {transactions_per_customer.median():.0f}")
print(f"   Max:    {transactions_per_customer.max()}")

# Revenue per customer
revenue_per_customer = df_clean.groupby('CustomerID')['TotalAmount'].sum()
print(f"\nRevenue per customer:")
print(f"   Mean (CLV):   {revenue_per_customer.mean():.2f}")
print(f"   Median (CLV): {revenue_per_customer.median():.2f}")
print(f"   Total:        {revenue_per_customer.sum():,.2f}")

# Top 5 customers by revenue
print(f"\nTop 5 customers by revenue:")
top_customers = revenue_per_customer.nlargest(5)
for i, (customer_id, revenue) in enumerate(top_customers.items(), 1):
    transactions = transactions_per_customer.loc[customer_id]
    print(f"   {i}. Customer {int(customer_id):5d}: {revenue:>10,.2f} ({transactions} transactions)")

# Top 5 countries by transaction count
print(f"\nTop 5 countries by transaction count:")
top_countries = df_clean['Country'].value_counts().head(5)
for country, count in top_countries.items():
    pct = (count / len(df_clean)) * 100
    print(f"   {country:20s}: {count:>7,} ({pct:5.1f}%)")

## 8. Cleaning Summary

### Data Quality Metrics

**Original Dataset:**
- 541,909 transactions
- 135,080 missing CustomerIDs (24.93%)
- 5,268 duplicates
- Negative values in Quantity and UnitPrice

**Cleaned Dataset:**
- 0 missing CustomerIDs
- 0 duplicates
- All positive values for Quantity and UnitPrice
- TotalAmount column added

**Rows Removed:**
- Rows without CustomerID (24.93%)
- Cancelled orders (InvoiceNo starting with 'C')
- Invalid quantities and prices
- Duplicate rows

---

### Key Insights

**Customer Distribution:**
- Average ~4 transactions per customer
- UK-based transactions dominate (~89%)

**Time Period:**
- Dec 1, 2010 to Dec 9, 2011 (12.4 months)

---

### Next Steps

The cleaned dataset is saved to `../data/online_retail_clean.csv` and is ready for
RFM analysis in `02_rfm_analysis.ipynb`. Outlier treatment will be applied at the
customer level after RFM aggregation.